# FPN-Mamba Experiments — A100 Colab Runner

**Before starting:** Make sure Runtime → Change runtime type → A100 GPU is selected.

**How persistence works:**
- **Code** comes from GitHub (pulled fresh each session via `git pull`)
- **Data, checkpoints, and results** live on Google Drive — they survive Colab resets
- If your session disconnects mid-training, re-run cells 1–3 to reconnect, then skip to whichever experiment wasn't finished yet

**Run order:**
1. Setup (cells 1–4)
2. InceptentionNet baseline (cell 5)
3. FPN-Mamba full model (cell 6)
4. Ablation table (cell 7)
5. Results & comparison (cells 8–9)

## Cell 1 — Verify A100 GPU

In [ ]:
import torch

assert torch.cuda.is_available(), 'No GPU found. Runtime → Change runtime type → GPU → A100'
gpu_name = torch.cuda.get_device_name(0)
vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9
bf16_ok  = torch.cuda.is_bf16_supported()

print(f'GPU  : {gpu_name}')
print(f'VRAM : {vram_gb:.1f} GB')
print(f'BF16 : {bf16_ok}  (True = A100/H100 — faster, more stable than FP16)')

if 'A100' not in gpu_name and 'H100' not in gpu_name:
    print('WARNING: Not an A100/H100. Batch sizes below may OOM — reduce if needed.')

## Cell 2 — Mount Google Drive

All data, checkpoints, and results are stored here.
Drive persists between Colab sessions; Colab's local `/content/` does not.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

# All experiment outputs live here on Drive
DRIVE_BASE    = '/content/drive/MyDrive/fpn_mamba'
DRIVE_DATA    = f'{DRIVE_BASE}/data'
DRIVE_RESULTS = f'{DRIVE_BASE}/experiments'

for d in [DRIVE_BASE, DRIVE_DATA, DRIVE_RESULTS]:
    os.makedirs(d, exist_ok=True)

print('Drive mounted.')
print(f'Data     -> {DRIVE_DATA}')
print(f'Results  -> {DRIVE_RESULTS}')

## Cell 3 — Pull Repo from GitHub & Install Dependencies

`git pull` fetches your latest code from **GitHub** (not from Drive or Google Cloud).
Drive only stores data and model outputs — code always comes from GitHub.

In [ ]:
import os, sys

REPO_URL  = 'https://github.com/Tech-sam-90/fpn-inceptentionnet'
REPO_DIR  = '/content/fpn-inceptentionnet'
BRANCH    = 'version_2'

if not os.path.exists(REPO_DIR):
    !git clone --branch {BRANCH} {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} fetch origin
    !git -C {REPO_DIR} checkout {BRANCH}
    !git -C {REPO_DIR} pull origin {BRANCH}

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
print('Repo ready at:', REPO_DIR)
!git log --oneline -5

In [ ]:
!pip install -q timm einops scikit-learn scipy thop pyyaml tqdm matplotlib seaborn pandas
print('Dependencies installed.')

## Cell 4 — Download Data (first time only)

Data is downloaded once to Drive and reused across sessions.
If `DRIVE_DATA/Meduloblastoma` already exists, this cell skips the download automatically.

**Kaggle API setup:** Upload your `kaggle.json` using Colab Files sidebar, or fill in credentials below.

In [ ]:
import os, glob

MB_CHECK = f'{DRIVE_DATA}/Meduloblastoma'

if os.path.exists(MB_CHECK):
    print(f'Data already on Drive at {DRIVE_DATA} — skipping download.')
    DATA_ROOT = DRIVE_DATA
else:
    print('Downloading dataset from Kaggle...')

    # Option 1: upload kaggle.json via Files sidebar, then run:
    os.makedirs('/root/.kaggle', exist_ok=True)
    !cp /content/kaggle.json /root/.kaggle/kaggle.json 2>/dev/null || true

    # Option 2: hardcode credentials (uncomment):
    # import json
    # with open('/root/.kaggle/kaggle.json', 'w') as f:
    #     json.dump({'username': 'YOUR_USERNAME', 'key': 'YOUR_API_KEY'}, f)

    !chmod 600 /root/.kaggle/kaggle.json
    !pip install -q kaggle
    !kaggle datasets download -d waseemnagahhenes/brain-tumor-for-14-classes \
        -p /tmp/kaggle_download --unzip

    # Move to Drive for persistence
    candidates = glob.glob('/tmp/kaggle_download/**/Meduloblastoma', recursive=True)
    if candidates:
        src = os.path.dirname(candidates[0])
        import shutil
        if os.path.exists(DRIVE_DATA):
            shutil.rmtree(DRIVE_DATA)
        shutil.copytree(src, DRIVE_DATA)
        print(f'Data copied to Drive: {DRIVE_DATA}')
    else:
        raise FileNotFoundError('Could not find Meduloblastoma folder in download.')

    DATA_ROOT = DRIVE_DATA

# Verify
classes = sorted(d for d in os.listdir(DATA_ROOT) if os.path.isdir(os.path.join(DATA_ROOT, d)))
print(f'\nData root : {DATA_ROOT}')
print(f'Classes   : {classes}')

## Config Helper

In [ ]:
import yaml
from pathlib import Path

def make_config(yaml_path: str, overrides: dict) -> str:
    """
    Load a YAML config, apply overrides (nested dict), write to /tmp.
    Returns the patched config path.
    """
    def _deep_update(base, patch):
        for k, v in patch.items():
            if isinstance(v, dict) and k in base:
                _deep_update(base[k], v)
            else:
                base[k] = v

    with open(yaml_path) as f:
        cfg = yaml.safe_load(f)

    _deep_update(cfg, overrides)

    out = f'/tmp/{Path(yaml_path).stem}_patched.yaml'
    with open(out, 'w') as f:
        yaml.dump(cfg, f)
    return out

# A100-specific overrides applied to ALL experiments
A100_BASE = {
    'data': {'data_root': DATA_ROOT},
    'training': {'num_workers': 4},     # A100 instances have ≥12 CPU cores
}

print('Config helper ready.')

## Cell 5 — Train InceptentionNet Baseline

Uses paper-exact hyperparameters: LR=0.005, batch=8, 40 epochs, patience=10, Gaussian σ=2.0, 106 MB images, 4× augmentation.
Batch size stays at 8 to faithfully replicate the paper — changing it would confound the comparison.

In [ ]:
import os

BASELINE_RUN_DIR = f'{DRIVE_RESULTS}/inceptentionnet'
BASELINE_RESULTS = f'{BASELINE_RUN_DIR}/cv_results.json'

if os.path.exists(BASELINE_RESULTS):
    print(f'Baseline results already exist at {BASELINE_RESULTS}')
    print('Delete that file and re-run this cell if you want to retrain.')
else:
    baseline_cfg = make_config(
        f'{REPO_DIR}/configs/inceptentionnet.yaml',
        overrides={
            **A100_BASE,
            'output': {'run_dir': BASELINE_RUN_DIR},
        }
    )
    !python scripts/train.py --config {baseline_cfg}

print('\nBaseline done. Results:', BASELINE_RESULTS)

## Cell 6 — Train FPN-Mamba (Full Model)

A100-specific: batch=32 (vs 16 on T4), num_workers=4, bfloat16 activated automatically by the trainer.

In [ ]:
import os

FPN_RUN_DIR = f'{DRIVE_RESULTS}/fpn_mamba'
FPN_RESULTS = f'{FPN_RUN_DIR}/cv_results.json'

if os.path.exists(FPN_RESULTS):
    print(f'FPN-Mamba results already exist at {FPN_RESULTS}')
    print('Delete that file and re-run this cell if you want to retrain.')
else:
    fpn_cfg = make_config(
        f'{REPO_DIR}/configs/fpn_mamba.yaml',
        overrides={
            **A100_BASE,
            'training': {'batch_size': 32, 'num_workers': 4},  # A100: 40GB VRAM, safe at 32
            'output':   {'run_dir': FPN_RUN_DIR},
        }
    )
    !python scripts/train.py \
        --config {fpn_cfg} \
        --baseline_results {BASELINE_RESULTS}

print('\nFPN-Mamba done. Results:', FPN_RESULTS)

## Cell 7 — Ablation Study

Trains all 5 variants. On A100 each takes ~20–30 min → ~2 hrs total.
Results auto-save to Drive after each variant, so a disconnect mid-run only loses the current variant.

In [ ]:
import os

ABL_RUN_DIR = f'{DRIVE_RESULTS}/ablation'
ABL_RESULTS = f'{ABL_RUN_DIR}/ablation_summary.json'

abl_cfg = make_config(
    f'{REPO_DIR}/configs/ablation.yaml',
    overrides={
        **A100_BASE,
        'training': {'batch_size': 32, 'num_workers': 4},
        'output':   {'run_dir': ABL_RUN_DIR},
    }
)

# To run a subset only: add --variants efficientnet_only fpn_standard
!python scripts/run_ablation.py --config {abl_cfg}

print('\nAblation done. Summary:', ABL_RESULTS)

## Cell 8 — Statistical Comparison (Baseline vs FPN-Mamba)

Reads saved JSON — no retraining. Shows bootstrap 95% CI and Wilcoxon p-values.

In [ ]:
import json
from src.evaluation.stats import compare_models, print_comparison_table
from src.evaluation.metrics import summarize_folds

with open(BASELINE_RESULTS) as f: baseline = json.load(f)
with open(FPN_RESULTS)      as f: fpn      = json.load(f)

print('=== InceptentionNet (re-run, paper-exact settings) ===')
for k, v in summarize_folds(baseline['fold_results']).items():
    print(f'  {k:<18}: {v["mean"]:.4f} +/- {v["std"]:.4f}')

print('\n=== FPN-Mamba ===')
for k, v in summarize_folds(fpn['fold_results']).items():
    print(f'  {k:<18}: {v["mean"]:.4f} +/- {v["std"]:.4f}')

print('\n=== Statistical Comparison (bootstrap 95% CI + Wilcoxon p) ===')
table = compare_models(
    baseline['fold_results'], fpn['fold_results'],
    name_a='InceptentionNet', name_b='FPN-Mamba'
)
print_comparison_table(table, name_a='InceptentionNet', name_b='FPN-Mamba')

## Cell 9 — Ablation Table

In [ ]:
import json, pandas as pd, numpy as np

with open(ABL_RESULTS) as f:
    abl = json.load(f)

DISPLAY = {
    'efficientnet_only': 'EfficientNet-B2 only',
    'fpn_standard':      '+ FPN (standard 3x3)',
    'fpn_locality':      '+ LocalityMixing',
    'fpn_cross_mamba':   '+ Cross-scale Mamba',
    'fpn_mamba_full':    '+ GeM + SE  (full model)',
}
METRICS = ['accuracy', 'precision', 'recall', 'sensitivity', 'specificity', 'f1', 'auc']

rows = []
for variant, name in DISPLAY.items():
    if variant not in abl: continue
    row = {'Variant': name}
    for m in METRICS:
        mu  = abl[variant].get(m, {}).get('mean', float('nan'))
        std = abl[variant].get(m, {}).get('std',  float('nan'))
        row[m] = f'{mu:.4f} ± {std:.4f}'
    rows.append(row)

df = pd.DataFrame(rows).set_index('Variant')
print('=== ABLATION TABLE ===')
print(df.to_string())

## Cell 10 — Verify Everything is on Drive

In [ ]:
import os

def check_file(path, label):
    exists = os.path.exists(path)
    size   = f'{os.path.getsize(path)/1024:.1f} KB' if exists else ''
    status = '[OK]' if exists else '[MISSING]'
    print(f'  {status} {label:<35} {size}  {path}')

print('=== Drive Contents ===')
check_file(BASELINE_RESULTS, 'InceptentionNet cv_results.json')
check_file(FPN_RESULTS,      'FPN-Mamba cv_results.json')
check_file(ABL_RESULTS,      'Ablation summary.json')

# Fold checkpoints
for run_name, run_dir in [('Baseline', BASELINE_RUN_DIR), ('FPN-Mamba', FPN_RUN_DIR)]:
    for fold in range(1, 6):
        check_file(f'{run_dir}/fold_{fold}.pt', f'{run_name} fold_{fold}.pt')
    check_file(f'{run_dir}/best_model.pt', f'{run_name} best_model.pt')

print('\nAll outputs are on Google Drive and will survive Colab disconnects.')